# M5 Re-ID 可行性驗證(Market-1501)— DINOv2(可商用) vs OSNet(對照)

量 Rank-1/mAP + chef_id 綁定正確率,判斷**可商用的 DINOv2** 夠不夠用。

**先備**:GPU(T4);Market-1501 資料集(研究用途);上傳我們的程式碼。

⚠ Market-1501 與 OSNet 權重皆**研究限定,僅供驗證、不可出貨**;可出貨路徑=DINOv2 或自訓。

In [ ]:
# 1) 安裝(Colab 已有 torch;裝 torchreid 供 OSNet 對照)
!pip -q install torchreid gdown
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# 2) 上傳程式碼:本機先打包 → zip -r m5eval.zip src/m5_reid scripts/reid_eval_market1501.py
from google.colab import files
print('請上傳 m5eval.zip')
files.upload()
!unzip -q -o m5eval.zip -d /content/m5eval
!ls /content/m5eval/src/m5_reid /content/m5eval/scripts

In [ ]:
# 3) 取得 Market-1501(研究用途,擇一)並解壓到 /content/market1501
#    (a) 上傳 market1501.zip:  from google.colab import files; files.upload()
#    (b) Kaggle:  !kaggle datasets download -d pengcw1/market-1501 (需 kaggle.json)
# 解壓後確認有 query/ 與 bounding_box_test/:
!ls /content/market1501 || echo '請先放入 Market-1501(含 query/、bounding_box_test/)'

In [ ]:
# 4) DINOv2(可商用候選)。加 --max-ids 100 可先快速試跑
!cd /content/m5eval && python scripts/reid_eval_market1501.py \
    --data /content/market1501 --embedder dinov2 --model-name dinov2_vits14 \
    --out /content/reid_dinov2.json

In [ ]:
# 5) OSNet(研究上限對照,不可出貨)
!cd /content/m5eval && python scripts/reid_eval_market1501.py \
    --data /content/market1501 --embedder osnet --model-name osnet_x1_0 \
    --out /content/reid_osnet.json

In [ ]:
# 6) 對照 + 下載
import json
for name, p in [('DINOv2(可商用)', '/content/reid_dinov2.json'), ('OSNet(對照)', '/content/reid_osnet.json')]:
    d = json.load(open(p, encoding='utf-8'))
    print(name, '→ Rank-1', d['cmc_map']['rank1'], 'mAP', d['cmc_map']['mAP'],
          '| 綁定正確', d['best_binding']['accuracy'], '@門檻', d['best_binding']['thr'])
from google.colab import files
files.download('/content/reid_dinov2.json'); files.download('/content/reid_osnet.json')

## 判讀
- OSNet Rank-1 對上發表(~0.94)→ 抽特徵+比對流程正確。
- **DINOv2 數字 = 可商用選項真實水準**;與 OSNet 差距 = 用可出貨模型的代價。
- DINOv2 綁定正確率夠高 → 可商用 M5 可行;差太多 → 走自訓。
- ⚠ 街上≠廚房制服 → 最終仍需自錄廚房多人資料驗證。把兩個 json 貼回給 Claude 判讀。